# Basketball Broadcast Detection and BEV Mini-map Pipeline

This notebook is the main entry point for the coursework implementation. It prepares the YOLO dataset, trains or loads a YOLOv8n detector, evaluates the detector, computes a manually calibrated homography, and renders `data/Demo Video.mp4` with detection, tracking, and a top-right mini-map overlay.

## 1. Setup and Paths

All paths and experiment parameters are loaded from `configs/project.yaml`. Update `raw_data_root` before running the notebook on the remote Windows machine.

In [ ]:
from pathlib import Path

from src.config import load_project_config, configured_path
from src.dataset import prepare_yolo_dataset, dataset_statistics
from src.train_yolo import train_yolo, best_weights_path
from src.evaluate_yolo import evaluate_standard_and_per_clip
from src.homography import (
    load_homography_config,
    compute_homography_dlt,
    reprojection_errors,
    draw_reference_points,
)
from src.minimap import court_to_pixel, draw_court_template, standard_landmarks_m
from src.video import render_demo_video

config = load_project_config("configs/project.yaml")
outputs_root = configured_path(config, "outputs_root")
outputs_root.mkdir(parents=True, exist_ok=True)

config

## 2. Dataset Inspection and Split

`clip1` is split by time order: the first 400 frames are used for training and the last 100 frames are used for validation. `clip2`, `clip3`, and `clip4` are used as the test set.

In [ ]:
# Run once after updating raw_data_root. Set overwrite=True if the processed dataset must be rebuilt.
PREPARE_DATASET = False

if PREPARE_DATASET:
    split_summary = prepare_yolo_dataset(config, overwrite=True)
    print(split_summary["data_yaml"])

stats = dataset_statistics(config)
stats

## 3. YOLO Training / Loading Trained Model

The baseline detector is `yolov8n.pt`, trained for 50 epochs at image size 640. Custom augmentation is disabled for the main run.

In [ ]:
# Keep this disabled when you only want to reuse an existing trained model.
RUN_TRAINING = False

if RUN_TRAINING:
    train_info = train_yolo(config)
    model_path = Path(train_info["best_weights"])
else:
    model_path = best_weights_path(config)

model_path

## 4. Detection Evaluation

Validation metrics are computed on the last 100 frames of `clip1`. Test metrics are computed on `clip2 + clip3 + clip4`, with optional per-clip results.

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    evaluation_results = evaluate_standard_and_per_clip(config, model_path)
    evaluation_results


## 5. Manual Homography Calibration

Camera B is calibrated from a clear `clip2` frame. The image points are manually selected once and saved in `configs/homography_camera_b.json`. The corresponding template points are specified as standard full-court metre coordinates and converted to mini-map pixel coordinates.

In [ ]:
homography_config = load_homography_config(config["paths"]["homography_config"])

# Useful names for selecting template_points_m in configs/homography_camera_b.json.
standard_landmarks_m()

In [ ]:
image_points = homography_config["image_points_px"]
template_points_m = homography_config["template_points_m"]

template_points_px = court_to_pixel(
    template_points_m,
    template_size=tuple(homography_config["template_size"]),
    court_size_m=tuple(homography_config["court_size_m"]),
    margin=int(homography_config["court_margin_px"]),
)

H = compute_homography_dlt(image_points, template_points_px, normalize=True)
calibration_error = reprojection_errors(image_points, template_points_px, H)

H, calibration_error

## 6. Mini-map Template Preview

In [ ]:
import matplotlib.pyplot as plt

template = draw_court_template(
    template_size=tuple(homography_config["template_size"]),
    margin=int(homography_config["court_margin_px"]),
)
plt.figure(figsize=(10, 5))
plt.imshow(template[..., ::-1])
plt.axis("off");

## 7. Demo Video Generation

The final demo renders `data/Demo Video.mp4` with detection boxes, compressed player/referee IDs, short trajectories, ball projection with short missing-frame tolerance, and a top-right mini-map overlay.

In [ ]:
RUN_DEBUG_DEMO = False
RUN_FULL_DEMO = False

demo_video_path = Path(config["paths"]["demo_video_path"])
demo_dir = outputs_root / "demo"
snapshot_dir = outputs_root / "screenshots"

if RUN_DEBUG_DEMO:
    debug_result = render_demo_video(
        config=config,
        model_path=model_path,
        video_path=demo_video_path,
        homography=H,
        output_video_path=demo_dir / "demo_video_debug_150.mp4",
        frame_limit=int(config["visualization"]["frame_limit_debug"]),
        snapshot_indices={20, 80},
        snapshot_dir=snapshot_dir,
    )
    print(debug_result)

if RUN_FULL_DEMO:
    full_result = render_demo_video(
        config=config,
        model_path=model_path,
        video_path=demo_video_path,
        homography=H,
        output_video_path=demo_dir / "demo_video_full.mp4",
        frame_limit=None,
        snapshot_indices={50, 150, 280, 420},
        snapshot_dir=snapshot_dir,
    )
    print(full_result)

## 8. Results and Discussion

Use the saved metrics JSON files, demo video, two success screenshots, and two failure screenshots for the final report. The critical analysis should discuss occlusion, small-ball detection, foot-point uncertainty, airborne objects, limited reference points, slight camera movement, and lighting or shadow effects.